# Flood Mapping with Sentinel-1 SAR Imagery

## Introduction
This notebook demonstrates a workflow for detecting flood extent using Sentinel-1 Synthetic Aperture Radar (SAR) imagery and Google Earth Engine (GEE). The process involves loading pre-event and post-event SAR data, calculating a difference image, applying a threshold to identify flooded areas, and visualizing the results on an interactive map.

In [3]:
# Flood Mapping with Sentinel-1 SAR
# Author: Florence Owiti
# Objective: Detect flood extent using open radar imagery

# ----------------------------------------------------------------------------------
# Step 1: Import Libraries
# ----------------------------------------------------------------------------------
!pip install earthengine-api geemap
import ee # Google Earth Engine Python API
import geemap # For visualization
import matplotlib.pyplot as plt

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-florenceaowiti')

## 1. Setup and Earth Engine Initialization
This section handles the necessary library installations and imports, along with authenticating and initializing the Google Earth Engine API. This is a crucial first step for accessing and processing satellite imagery data.

In [4]:

# ---------------------------------------------------------------------------------
# Step 2: Define Region of Interest (ROI)
# ---------------------------------------------------------------------------------
# Example: Flood-prone area in Kenya
roi = ee.Geometry.Polygon([
    [[34.5, -0.1], [34.9, -0.1], [34.9, 0.3], [34.5, 0.3]]
])


## 2. Define Region of Interest (ROI)
Here, we define a specific geographical area, known as the Region of Interest (ROI), where we want to detect floods. For this example, we've selected a flood-prone area in Kenya using `ee.Geometry.Polygon`.

In [5]:
# ----------------------------------------------------------------------------------
# Step 3: Load Sentinel-1 SAR Data
# ----------------------------------------------------------------------------------
# Prevent-event image
pre_event = (ee.ImageCollection('COPERNICUS/S1_GRD')
             .filterBounds(roi)
             .filterDate('2020-04-01', '2020-04-10')
             .filter(ee.Filter.eq('instrumentMode', 'IW'))
             .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
             .select('VV')
             .mean())


## 3. Load and Filter Sentinel-1 SAR Data
Sentinel-1 SAR data is used because it can penetrate clouds and works regardless of sunlight, making it ideal for flood detection. We retrieve two sets of images: 'pre-event' (before the flood) and 'post-event' (after the flood), filtering by dates, instrument mode, and polarization.

In [7]:
# Post-event image
post_event = (ee.ImageCollection('COPERNICUS/S1_GRD')
              .filterBounds(roi)
              .filterDate('2020-04-10', '2020-04-20') # Date range after the flood event
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
              .select('VV')
              .mean())

In [11]:
# ----------------------------------------------------------------------------------
# Step 4: Flood Detection (Difference Image)
# ----------------------------------------------------------------------------------
# Difference image
diff = post_event.subtract(pre_event)

## 4. Flood Detection using Difference Imaging
To identify flooded areas, we calculate the difference between the post-event and pre-event SAR images. Flooded areas typically show a significant decrease in radar backscatter. A threshold is applied to this difference image to delineate the flood extent.

In [12]:
# ----------------------------------------------------------------------------------
# Step 4: Flood Detection (Flood Extent)
# ----------------------------------------------------------------------------------
# Calculate flood extent using a threshold
# You might need to adjust this threshold based on your specific area and data.
threshold = -2.5 # Example threshold in dB
flood = diff.lt(threshold).selfMask()

In [14]:
# ----------------------------------------------------------------------------------
# Step 5: Visualization
# ----------------------------------------------------------------------------------
Map = geemap.Map(center=[0.1, 34.7], zoom=10)
Map.addLayer(pre_event, {'min': -20, 'max': 0}, 'Pre-event VV')
Map.addLayer(post_event, {'min': -20, 'max': 0}, 'Post-event VV')
Map.addLayer(flood, {'palette': ['blue']}, 'Flood Extent')
Map

Map(center=[0.1, 34.7], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

## 5. Visualize Flood Extent
Finally, the detected flood extent, along with the pre-event and post-event SAR images, are visualized on an interactive map using `geemap`. This allows for a clear and intuitive understanding of the flood's impact.

## Conclusion
This notebook successfully demonstrates how Sentinel-1 SAR data can be leveraged with Google Earth Engine and `geemap` to perform flood mapping. The resulting interactive map provides a valuable tool for disaster response and environmental monitoring. Further analysis could involve refining the threshold, incorporating other SAR bands, or integrating additional geospatial data for improved accuracy.